In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

def compute_and_save_embeddings(input_file, output_file, client, embedding_model=EMBEDDING_MODEL, text_column=TEXT_COLUMN):
    df = pd.read_csv(input_file)

    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found in {input_file}. Columns: {list(df.columns)}")

    def get_embedding(text: str, model=embedding_model):
        if pd.isna(text):
            return None
        text = str(text).replace("\n", " ")
        try:
            resp = client.embeddings.create(model=model, input=[text])
            return resp.data[0].embedding
        except Exception as e:
            print(f"[ERROR] {input_file}: {e}")
            return None

    tqdm.pandas(desc=f"Embeddings: {os.path.basename(input_file)}")
    df["embedding"] = df[text_column].progress_apply(get_embedding)

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

def run_folder(input_folder=INPUT_FOLDER, output_folder=OUTPUT_FOLDER, recursive=False):
    api_key = os.environ.get("OPENAI_API_KEY", "OPENAI_API_KEY")

    if not api_key:
        raise EnvironmentError("Set OPENAI_API_KEY as an environment variable (don't hardcode it).")

    client = OpenAI(api_key=api_key)
    os.makedirs(output_folder, exist_ok=True)

    if recursive:
        walker = []
        for root, _, files in os.walk(input_folder):
            for f in files:
                if f.lower().endswith(".csv"):
                    walker.append(os.path.join(root, f))
        csv_files = walker
    else:
        csv_files = [
            os.path.join(input_folder, f)
            for f in os.listdir(input_folder)
            if f.lower().endswith(".csv")
        ]

    for input_path in csv_files:
        base = os.path.splitext(os.path.basename(input_path))[0]
        output_path = os.path.join(output_folder, f"{base}_embedded_text.csv")
        compute_and_save_embeddings(input_path, output_path, client=client)

In [ ]:
INPUT_FOLDER = "Data"
OUTPUT_FOLDER = "Embeddings"
EMBEDDING_MODEL = "text-embedding-3-large"
TEXT_COLUMN = "question"

if __name__ == "__main__":
    run_folder(INPUT_FOLDER, OUTPUT_FOLDER, recursive=False)
